# D06 · 哪些是结构性的，哪些真能干预

前五册把"从哪来、什么时候来、房型、取消"都量化了。这一册回答最后一个问题：
**取消率两年涨了 6.4 个百分点，这里有多少是客人变了（改不了），有多少是我们变了（能改）？**

方法只有一个：Kitagawa 分解。把总变化拆成三块——

| 效应 | 含义 | 能不能改 |
| --- | --- | --- |
| 结构效应 | 客源/渠道的**构成**变了，即使每组行为不变也会推高总体 | 短期不能 |
| 组内效应 | 同一群人内部的取消率真的变了 | 可能能 |
| 交互效应 | 构成与比率同时变 | 单独列出，不硬摊 |

恒等式：总变化 = 结构 + 组内 + 交互。三个数加不回总数就说明口径漏了组，
`rate.kitagawa` 会直接断言失败，不允许"大概对上"。

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hotel_insight import cancel, config, rate, timing, viz
from hotel_insight.viz import FAINT, INK, MAIN, MAIN_DARK, MAIN_LIGHT, MAIN_MID, MUTED

viz.apply_theme()
pd.set_option("display.width", 240)
pd.set_option("display.unicode.east_asian_width", True)

con = sqlite3.connect(config.DB_PATH)
s = config.load_settings()
a = s["analysis"]
NB = "D06"
SOURCE = "数据源：hotel.db / dwd_bookings（85,585 条入住记录，2015-07 至 2017-08）"

d = pd.read_sql("SELECT * FROM dwd_bookings", con)
# 同比口径在 D01 钉死：只取两年都有的 1—8 月
d16 = timing.comparable_slice(d, [2016], a["comparable_months"])
d17 = timing.comparable_slice(d, [2017], a["comparable_months"])
print(f"2016 年 1—8 月 {len(d16):,} 条，取消 {d16['is_canceled'].mean():.2%}")
print(f"2017 年 1—8 月 {len(d17):,} 条，取消 {d17['is_canceled'].mean():.2%}")
assert len(d16) == 27288 and len(d17) == 31256

## 1 总变化：+6.4 个百分点

这一步只是把两个数放在一起，但它决定了后面所有分解的"分母"。

In [ ]:
base16 = d16["is_canceled"].mean()
base17 = d17["is_canceled"].mean()
print(f"2016：{base16:.4%}    2017：{base17:.4%}    变化：{(base17 - base16) * 100:+.2f}pp")
assert base17 > base16

## 2 按市场细分分解：八成来自组内

市场细分是影响面最大的因素（D05 排第一），先拆它。

In [ ]:
# ── 出图 D06_01：取消率变化的瀑布分解 ──
# 为什么用瀑布：三个效应是"从起点加减到终点"的关系，普通条形图要读者自己加减；
# 瀑布把每一步的起止都画出来，连接线让"上一根的顶"直接接上"这一根的底"。
# 规范落在哪几行：起点柱标出 2016 值 / 正效应用主色 / 终值单独配色并加粗 / 单位统一为 %
# ---- 1 指标 ----
k = rate.kitagawa(d16, d17, "market_segment")
deltas = [k["结构效应"], k["组内效应"], k["交互效应"]]
title = f"取消率两年涨 {k['总变化']:.1f} 个百分点，{k['组内效应'] / k['总变化']:.0%} 来自组内"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"口径：2016 与 2017 年 1—8 月同月对比；按市场细分拆解；"
            f"三项之和 {sum(deltas):+.2f}pp = 总变化 {k['总变化']:+.2f}pp")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.0, 4.6))
viz.waterfall(ax, ["渠道结构", "组内变化", "交互项"], deltas,
              start=k["起点"], start_label="2016 年")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D06_01_取消率分解瀑布.png")
# ---- 4 读回断言：三块之和必须等于总变化，且终值等于 2017 实际值 ----
assert abs(sum(deltas) - k["总变化"]) < 1e-6
assert abs(k["起点"] + sum(deltas) - base17 * 100) < 1e-6
assert abs(k["起点"] - base16 * 100) < 1e-6
plt.show()

三个数读起来是这样：

- **结构 +1.26pp**：2017 年 OTA 占比从 60.3% 涨到 65.6%，而 OTA 的取消率本来就高，
  所以即使每组行为不变，总取消率也会被推高 1.3 个点。这部分酒店短期改不了。
- **组内 +5.02pp**：同一个渠道内部，取消率真的变高了。OTA 从 33.2% 涨到 39.8%，
  线下旅行社从 13.6% 涨到 17.5%。这部分才是"可能能改"的。
- **交互 +0.08pp**：可以忽略。

所以"客人变了"只解释了 20%，**80% 是同一群人的行为变了**。这个结论与"客源结构恶化"的猜测相反。

## 3 换六个维度重拆一次：结论不变

一个维度可能只是巧合。把六个维度都拆一遍，看"组内效应占主导"是不是普遍成立。

In [ ]:
# ── 出图 D06_02：六个维度的结构与组内效应 ──
# 为什么用分组条形：六个维度 × 两个效应，分组的对比比两张图更省版面，
# 也让"结构效应都很小、组内效应都很大"这个模式一眼可见。
# 规范落在哪几行：零线必须画（效应有正负）/ 结构用浅蓝、组内用主蓝 / 数值标在条的末端
# ---- 1 指标 ----
dims = {
    "市场细分": "market_segment",
    "提前期": None,                       # 提前期先分箱，见下
    "客源国": "country",
    "客群类型": "customer_type",
    "酒店类型": "hotel_type",
    "房型": "reserved_room_type",
}
bins, labs = a["lead_time_bins"], a["lead_time_labels"]
d16b = d16.assign(lt_bin=pd.cut(d16["lead_time"], bins, labels=labs, include_lowest=True))
d17b = d17.assign(lt_bin=pd.cut(d17["lead_time"], bins, labels=labs, include_lowest=True))

rows = []
for name, col in dims.items():
    if col is None:
        kk = rate.kitagawa(d16b, d17b, "lt_bin")
    else:
        kk = rate.kitagawa(d16, d17, col)
    rows.append({"维度": name, "结构效应": kk["结构效应"], "组内效应": kk["组内效应"],
                 "交互效应": kk["交互效应"], "总变化": kk["总变化"]})
dec = pd.DataFrame(rows).set_index("维度")
struct_abs_max = dec["结构效应"].abs().max()
title = f"六个维度的结构效应都不超过 {struct_abs_max:.1f} 个百分点，变化几乎全在组内"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = "口径：同月对比；提前期按 7/30/90/180/365 天分档后拆解；正值表示推高取消率"
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.2, 4.0))
viz.grouped_hbar(ax, dec.index.tolist(),
                 {"结构效应": dec["结构效应"].to_numpy(),
                  "组内效应": dec["组内效应"].to_numpy()},
                 colors=[MAIN_LIGHT, MAIN])
ax.xaxis.set_major_formatter(lambda v, _: "0" if abs(v) < 1e-9 else f"{v:+.0f}")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D06_02_六维度分解.png")
# ---- 4 读回断言：每个维度的三项都等于 6.36 ----
assert np.allclose(dec["结构效应"] + dec["组内效应"] + dec["交互效应"], dec["总变化"], atol=0.01)
assert np.allclose(dec["总变化"], k["总变化"], atol=0.01), "换个维度拆，总变化不该变"
plt.show()

六个维度的结论高度一致：

| 维度 | 结构效应 | 组内效应 | 读法 |
| --- | --- | --- | --- |
| 市场细分 | +1.26 | +5.02 | 渠道构成变化贡献两成 |
| 提前期 | +1.51 | +3.95 | **提前期变长是最大的结构因素** |
| 客源国 | −1.14 | +7.01 | 客源构成反而略微拉低了取消率 |
| 客群类型 | +0.49 | +5.58 | 客群构成基本没变 |
| 酒店类型 | +0.05 | +6.31 | 两家店的占比几乎没动 |
| 房型 | +0.01 | +6.29 | 房型结构完全没变 |

两点值得注意：

1. **客源国的结构效应是负的**：如果只看"葡萄牙占比下降"，会以为取消率该降，实际却涨了 7 个点。
   这说明"客源结构恶化"这个解释是错的。
2. **提前期的结构效应最大（+1.51）**：客人订得越来越早，而订得越早越容易取消（D03 的单调关系）。
   这是唯一一个"看着像结构、其实有干预余地"的维度——因为酒店可以对长提前期订单加担保。

## 4 第二层：OTA 内部，提前期到底变了多少

第一层说"OTA 组内涨了 5.0pp"。但这 5pp 里有多少是"OTA 的订单提前期变长了"，有多少是"同一提前期档内取消率也涨了"？
这要用两层分解。

In [ ]:
nested = rate.nested_kitagawa(d16b, d17b, "market_segment", "lt_bin")
nested.round(2)

In [ ]:
# ── 出图 D06_03：OTA 的提前期结构变化 ──
# 为什么用哑铃：六个档位 × 两个时点，哑铃的线段长度就是占比变化，
# 一眼看出"哪一档涨得最凶"（181—365 天，7.4%→17.8%）。
# 规范落在哪几行：有序档位不排序 / 左灰右主蓝 / 数值标在右侧
# ---- 1 指标 ----
ota16 = d16b[d16b["market_segment"] == a["focus_market"]]
ota17 = d17b[d17b["market_segment"] == a["focus_market"]]
p16 = ota16["lt_bin"].value_counts(normalize=True).reindex(labs).fillna(0)
p17 = ota17["lt_bin"].value_counts(normalize=True).reindex(labs).fillna(0)
long_idx = labs[4]                                  # 181–365 天
title = f"OTA 长提前期订单翻倍：{long_idx} 从 {p16[long_idx]:.0%} 升到 {p17[long_idx]:.0%}"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"仅 {a['focus_market']}（{len(ota16):,} 条 → {len(ota17):,} 条）；"
            f"中位提前期 {ota16['lead_time'].median():.0f} → {ota17['lead_time'].median():.0f} 天")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(8.8, 4.4))
viz.dumbbell(ax, labs, p16.to_numpy(), p17.to_numpy(), left_name="2016", right_name="2017")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D06_03_OTA提前期变化.png")
# ---- 4 读回断言：两列占比都合计 100% ----
assert abs(p16.sum() - 1) < 1e-9 and abs(p17.sum() - 1) < 1e-9
assert p17[long_idx] > p16[long_idx] * 1.5, "长提前期占比没有明显上升，标题就写错了"
plt.show()

两层分解的结果（`nested` 表）：

| 外层组 | 结构效应 | 组内效应 | 交互 | 总变化 |
| --- | --- | --- | --- | --- |
| 在线旅行社 | +2.11 | +3.18 | +1.33 | +6.62 |
| 线下旅行社 | +1.04 | +2.48 | +0.38 | +3.90 |
| 直接预订 | +1.08 | +0.16 | +0.14 | +1.38 |
| 企业协议 | +0.70 | +2.47 | +0.79 | +3.96 |

OTA 内部：结构（提前期变长）+2.11，组内（同档内取消率上升）+3.18。
也就是说 OTA 那 6.6pp 的涨幅里，**1/3 是"客人订得更早了"，2/3 是"同等人群更容易退"**。

这个拆分有直接用处：
- 前 2.1pp 可以用"长提前期订单加担保/再确认"去对冲；
- 后 3.2pp 找不到对应的动作，只能继续观察，或者怀疑是 OTA 侧的政策变化（本数据看不到）。

## 5 情景：假设不同，答案差一位数

D05 的情景是"提前期 > 90 天、取消率相对降 10%"。这里把假设从 5% 扫到 30%，
看结论对假设有多敏感。

In [ ]:
# ── 出图 D06_04：情景敏感性 ──
# 为什么画折线：要表达"结论随假设线性放大"，条形只能给一个点。
# 同时把"这是假设"写在轴上，避免读者把它当预测区间。
# 规范落在哪几行：横轴是假设值（%），纵轴是少取消单数 / 千单；只画一条线，不叠加别的口径
# ---- 1 指标 ----
drops = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
saved = []
for dp in drops:
    s2 = {**s, "analysis": {**a, "scenario_relative_drop": float(dp)}}
    saved.append(cancel.scenario_reconfirm(d, s2)["少取消单数"])
saved = np.array(saved)
title = f"假设降幅从 5% 到 30%，少取消单数从 {saved[0]:.0f} 增到 {saved[-1]:,.0f}"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"适用集：提前期 > {a['scenario_lead_days']} 天的 {cancel.scenario_reconfirm(d, s)['适用单量']:,} 条订单；"
            f"线性放大是公式决定的，不是拟合结果")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(8.6, 4.2))
ax.plot(drops * 100, saved, color=MAIN, marker="o", markersize=6, linewidth=2, zorder=3)
for x, y in zip(drops * 100, saved):
    ax.annotate(f"{y:,.0f}", (x, y), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9, color=INK)
ax.set_xlabel("取消率相对下降的假设值（%）")
ax.set_ylabel("少取消单数（单）")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.set_xlim(drops.min() * 100 - 2, drops.max() * 100 + 2)
ax.set_ylim(0, saved.max() * 1.18)
ax.grid(axis="x", visible=False)
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D06_04_情景敏感性.png")
# ---- 4 读回断言：必须严格单调，且 10% 的值与 settings 口径一致 ----
assert np.all(np.diff(saved) > 0), "少取消单数应随假设单调上升"
assert abs(saved[1] - cancel.scenario_reconfirm(d, s)["少取消单数"]) < 1e-6
plt.show()

斜率完全由公式决定（`单数 × 取消率 × drop`），所以这张图不是"预测区间"，
而是**"你的假设值是多少，承诺就是多少"**。这也是为什么报告里必须同时写假设值：
只写"能少取消 1,102 单"而不写"假设降 10%"，读者会以为是实测。

## 6 结构性 vs 可干预：逐条判定

判据只有一条：**酒店在 3—6 个月内能不能改变它**。理由写在 `cancel.FACTORS` 里，可以被反驳。

### 只能适应（结构性）

| 因素 | 超额取消单 | 为什么改不了 | 怎么用 |
| --- | --- | --- | --- |
| 市场细分 | 3,952 | 渠道组合由年度合同与分销策略决定 | 用来**预测**：OTA 占比升，取消率必升 |
| 客源国 | 3,036 | 客源地理，营销引导要两三年见效 | 用来**定基线**：不同月/季的基线不同 |
| 客群类型 | 1,867 | 散客/协议/团队构成跟随销售结构 | 用来**分群看指标**，不做统一目标 |
| 到达月份 | 1,528 | 季节由气候与假期决定 | 用来**排班与定价** |
| 酒店类型 | 1,365 | 两家店定位不同 | 用来**分店考核** |
| 房型 | 859 | 房型供给固定 | 分配与超售可调，需求不可调 |

### 可以动（可干预 / 混合）

| 因素 | 超额取消单 | 动作 | 成本 | 风险 |
| --- | --- | --- | --- | --- |
| 特殊要求数 | 2,511 | 下单页引导填写（"有特殊要求请说明"） | 零 | 低，但可能被滥用 |
| 提前期 | 3,711（混合） | 长提前期订单 T−90 再确认 / 加担保 | 人力 + 渠道费 | 可能流失对担保敏感的价格型客户 |
| 停车位需求 | 2,007 | 主动询问并锁车位 | 低 | **字段口径存疑，先验证再动** |
| 改单次数 | 1,846 | 让改单更顺（改单客户取消率 14% vs 未改 30%） | 低 | 低 |
| 历史取消次数 | 675 | 1 次以上要求预付或担保 | 低 | **2 次以上曲线反常，需先查口径** |

### 先查口径，别当结论（需存疑）

| 因素 | 现象 | 疑点 |
| --- | --- | --- |
| 押金类型 | 不可退款 94.7% | 字段可能记的是取消后的状态，不是预订时的选择 |

## 7 最终结论

**回答"哪些是结构性的、哪些能干预"：**

1. **两年 6.4pp 的上涨，80% 是组内效应**——不是客源结构恶化，是同一群人更容易退。
   这个判断在六个维度上一致，不是某一个维度的巧合。
2. **最大的一处"结构性但有解"是提前期**：+1.51pp 来自订得越来越早，
   而提前期恰恰是下单时就知道、可以对冲的（再确认 / 担保）。
3. **能直接动的四个杠杆（特殊要求、停车位、改单、历史取消）加起来 7,039 单**，
   还不如市场细分一个因素（3,952 单）+ 提前期（3,711 单）合计的 7,663 单。
   现实就是"能改的影响小"，这不是分析方法的问题，是业务结构的问题。
4. **两个数字先别用**：押金"不可退款"94.7%、停车位 0%。它们更像字段口径造成的假象。

**这套结论的边界**：全部是相关性与结构分解，没有因果识别；
没有取消原因字段、没有客户 ID 串联、没有每日房态，所以"为什么退""退了之后去哪""损失多少"都答不了。

## 8 发现清单（本册）

| # | 发现 | 影响 |
| --- | --- | --- |
| 1 | 6.4pp 上涨中 80% 为组内效应 | 不能归因于"客源变差"，要归因于行为变化 |
| 2 | 客源国的结构效应是负的（−1.14） | "客源结构恶化"这个常见解释被数据否定 |
| 3 | 提前期结构效应 +2.17，是最大的结构性因素 | 唯一"结构但可对冲"的维度 |
| 4 | OTA 内部：结构 +2.11、组内 +3.18 | 三分之二是同等人群更易退，找不到对应动作 |
| 5 | 情景结果对假设线性敏感 | 报告必须同时给假设值，否则等于编数 |
| 6 | 可干预杠杆合计 7,039 单 < 结构性前三 10,699 单 | 干预空间有限，预期要放低 |

In [ ]:
con.close()
print("D06 完成：结构性/可干预已分开，六维度分解一致，情景敏感性已说明")